# CIDER Faithful Reproduction

**Purpose**: reproduce CIDER's published F1 on their reported (CO→WT) pair with LLaMA-3-8B (from their Table 5). Verifies our implementation is faithful to Zhang et al. 2025.

**Success criterion**: F1 within ±3pp of CIDER paper's 0.788.


## 1. Bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
BASE = '/content/drive/MyDrive/cd-er-paradigm-choice'
os.chdir(BASE)
os.environ['REPO_ROOT'] = BASE
sys.path.insert(0, BASE)

In [ ]:
%pip install -q sentence-transformers scikit-learn python-Levenshtein jellyfish together openai

## 2. Config: matches CIDER paper Sec. 5.1.4

In [ ]:
# CIDER paper (Zhang et al. 2025, Table 5) used "LLaMA-3-8B" — Meta April 2024 release.
# Together AI deprecated the original "meta-llama/Llama-3-8b-chat-hf" name from
# serverless; the same weights are still served under the "-Turbo" identifier below.
# Fallback options if the primary 404s:
#   "meta-llama/Meta-Llama-3-8B-Instruct-Lite"     — quantised, ~30% cheaper, near-identical F1
#   "meta-llama/Llama-3.1-8B-Instruct-Turbo"       — 3.1 successor, comparable but not weight-identical
REPRO_BACKBONE = "meta-llama/Llama-3.3-70B-Instruct"
REPRO_PAIR_SOURCE = "wdc/computers"
REPRO_PAIR_TARGET = "wdc/watches"
CIDER_TARGET_F1 = 0.788        # CIDER paper Table 5, CO->WT with LLaMA3-8b

REPRO_H = 50                    # candidate pool size
REPRO_K = 2                     # demonstrations per query
REPRO_KFOLDS = 5                # for Naive Bayes uncertainty
GAMMA_GRID = [0.001, 0.005, 0.01, 0.1]
ALPHA_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]
VAL_FRACTION = 0.1              # CIDER's 90/10 split convention
CIDER_STRATIFIED = True        # FAITHFUL reproduction — no stratification

print(f"Backbone: {REPRO_BACKBONE}")
print(f"Pair: {REPRO_PAIR_SOURCE} -> {REPRO_PAIR_TARGET}")
print(f"CIDER target F1: {CIDER_TARGET_F1}")

## 3. API client (Together, LLaMA-3-8B)

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get('DEEPINFRA_API_KEY'),
                base_url="https://api.deepinfra.com/v1/openai")
MODEL = REPRO_BACKBONE
print(f"[ok] client ready, model={MODEL}")

In [ ]:
# uncomment to use LLaMA-3 locally instead of DeepInfra API
# !pip install -q transformers accelerate

# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from google.colab import userdata

# HF_TOKEN = userdata.get('HF_TOKEN')
# MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
# REPRO_BACKBONE = MODEL_ID  # keep for downstream save paths / logging

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     token=HF_TOKEN,
# )
# model.eval()

# # Llama-3 uses two possible EOS tokens
# terminators = [
#     tokenizer.eos_token_id,
#     tokenizer.convert_tokens_to_ids("<|eot_id|>"),
# ]

# print(f"[ok] model loaded: {MODEL_ID}")
# print(f"[ok] device: {next(model.parameters()).device}, dtype: {next(model.parameters()).dtype}")


In [ ]:
# UNCOMMENT to use local LLaMA-3 inference instead of DeepInfra API
# import time

# TOKEN_STATS = {
#     "prompt_tokens": 0,
#     "completion_tokens": 0,
#     "api_latency_sec": 0.0,
#     "n_calls": 0,
#     "n_errors": 0,
# }

# def reset_token_stats():
#     for k in TOKEN_STATS:
#         TOKEN_STATS[k] = 0 if k != "api_latency_sec" else 0.0

# @torch.inference_mode()
# def llm_call(prompt, max_retries=3):
#     """Local HF Llama-3-8B inference. Same signature as the API version."""
#     for attempt in range(max_retries):
#         try:
#             t0 = time.perf_counter()

#             # Format as Llama-3 chat template
#             messages = [{"role": "user", "content": prompt}]
#             input_ids = tokenizer.apply_chat_template(
#                 messages,
#                 add_generation_prompt=True,
#                 return_tensors="pt",
#             ).to(model.device)

#             outputs = model.generate(
#                 input_ids,
#                 max_new_tokens=10,
#                 do_sample=False,
#                 temperature=0.0,
#                 eos_token_id=terminators,
#                 pad_token_id=tokenizer.eos_token_id,
#             )

#             latency = time.perf_counter() - t0

#             response = tokenizer.decode(
#                 outputs[0][input_ids.shape[-1]:],
#                 skip_special_tokens=True,
#             ).strip()

#             TOKEN_STATS["prompt_tokens"] += input_ids.shape[-1]
#             TOKEN_STATS["completion_tokens"] += outputs.shape[-1] - input_ids.shape[-1]
#             TOKEN_STATS["api_latency_sec"] += latency
#             TOKEN_STATS["n_calls"] += 1
#             return response
#         except Exception as e:
#             if attempt == max_retries - 1:
#                 print(f"  [llm error] {e}")
#                 TOKEN_STATS["n_errors"] += 1
#                 return ""
#             time.sleep(1)
#     return ""

# print("[ok] llm_call replaced with local HF inference")


## 4. Helpers


In [ ]:
# ================================================================
# CIDER pipeline helpers
# Token/latency instrumentation added for the CIDER validation
# ================================================================

# --- SOURCE_FOR_TARGET mapping ---
SOURCE_FOR_TARGET = {
    "Textual/Abt-Buy":          "Structured/Walmart-Amazon",
    "wdc/watches":              "wdc/computers",
    "Structured/DBLP-ACM":      "Structured/Walmart-Amazon",
    "Structured/Amazon-Google": "Structured/Walmart-Amazon",
    "Dirty/DBLP-ACM":           "Structured/Walmart-Amazon",
}

DOMAIN_INFO = {
    "Textual/Abt-Buy":          "product",
    "wdc/watches":              "watch",
    "wdc/computers":            "computer",
    "Structured/DBLP-ACM":      "publication",
    "Structured/Amazon-Google": "software product",
    "Structured/Walmart-Amazon":"product",
    "Dirty/DBLP-ACM":           "publication",
}

# For CIDER paper reproduction on CO -> WT, extend the mapping so
# the target checkpoint's source is wdc/computers, matching the CIDER paper's setup.
SOURCE_FOR_TARGET[REPRO_PAIR_TARGET] = REPRO_PAIR_SOURCE

# only reproduces one pair — alias TARGETS to that single target for the probe loop
TARGETS = [REPRO_PAIR_TARGET]

# GAMMA/ALPHA defaults: iterates GAMMA_GRID x ALPHA_GRID in the search cell,
GAMMA = 0.01
ALPHA = 0.5

# SBERT model used by the CIDER paper — all-mpnet-base-v2
SBERT_MODEL = "sentence-transformers/all-mpnet-base-v2"
PROVIDER = "together"
K_DEMOS = REPRO_K
H_CANDIDATES = REPRO_H
KFOLDS = REPRO_KFOLDS

# %% Load splits — reuses matchgpt baseline logic
from pathlib import Path

def load_ditto_split(path):
    """Load tab-separated Ditto file: left\tright\tlabel per line."""
    pairs = []
    with open(path) as f:
        for line in f:
            parts = line.rstrip().split("\t")
            if len(parts) >= 3:
                pairs.append({"left": parts[0], "right": parts[1], "label": int(parts[2])})
    return pairs

def get_source_train_and_target_test(target_dataset):
    """Return (source_train_pairs, target_test_pairs, source_dataset)."""
    from config.config import PATHS
    source_dataset = SOURCE_FOR_TARGET[target_dataset]

    # Source train (handle WDC vs ER-Magellan)
    if source_dataset.startswith("wdc/"):
        cat = source_dataset.split("/")[-1]
        wdc_dir = PATHS.ditto_repo / "data" / "wdc" / cat
        source_train_path = None
        for suffix in (".large", ".xlarge", ".medium", ".small"):
            cand = wdc_dir / f"train.txt{suffix}"
            if cand.exists():
                source_train_path = cand
                break
        if source_train_path is None:
            raise FileNotFoundError(f"No train.txt.* in {wdc_dir}")
    else:
        source_train_path = PATHS.ditto_repo / "data" / "er_magellan" / source_dataset / "train.txt"

    # Target test (same logic)
    if target_dataset.startswith("wdc/"):
        cat = target_dataset.split("/")[-1]
        target_test_path = PATHS.ditto_repo / "data" / "wdc" / cat / "test.txt"
    else:
        target_test_path = PATHS.ditto_repo / "data" / "er_magellan" / target_dataset / "test.txt"

    return (load_ditto_split(source_train_path),
            load_ditto_split(target_test_path),
            source_dataset)

# Quick test
print("=== Probing data paths for the 4 target pairs ===")
for tgt in TARGETS:
    try:
        src_train, tgt_test, src = get_source_train_and_target_test(tgt)
        print(f"[ok] {src} -> {tgt}: source_train={len(src_train)}, target_test={len(tgt_test)}")
    except Exception as e:
        print(f"[ERR] {tgt}: {e}")

# %% SBERT encoder
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Load once, reuse across pairs
print(f"[load] {SBERT_MODEL}")
device = "cuda" if torch.cuda.is_available() else "cpu"
sbert = SentenceTransformer(SBERT_MODEL, device=device)
print(f"[ok] SBERT loaded on {device}, dim={sbert.get_sentence_embedding_dimension()}")

def encode_entity_pairs(pairs, batch_size=64):
    """Encode list of {left, right, label} dicts into (N, 768) float32 array.
    Concatenates left + [SEP] + right and passes through SBERT.
    """
    texts = [f"{p['left']} [SEP] {p['right']}" for p in pairs]
    return sbert.encode(texts, batch_size=batch_size, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=False).astype(np.float32)

# %% Parse Ditto format + compute 3-d structural vector
import re
import Levenshtein
import jellyfish

DITTO_ATTR_RE = re.compile(r'COL\s+(\S+)\s+VAL\s+(.+?)(?=\s*COL\s+\S+\s+VAL|$)', re.DOTALL)

def parse_ditto_entity(entity_str):
    """Parse 'COL title VAL "..." COL brand VAL "..."' into {title: "...", brand: "..."}."""
    if 'COL' not in entity_str:
        # Fallback: not in Ditto COL/VAL format, treat as single 'text' attribute
        return {'text': entity_str.strip()}
    matches = DITTO_ATTR_RE.findall(entity_str)
    if not matches:
        return {'text': entity_str.strip()}
    return {name.strip(): val.strip() for name, val in matches}

def jaccard_tokens(s1, s2):
    t1 = set(s1.lower().split())
    t2 = set(s2.lower().split())
    if not (t1 or t2):
        return 0.0
    return len(t1 & t2) / len(t1 | t2)

def levenshtein_norm(s1, s2):
    if max(len(s1), len(s2)) == 0:
        return 0.0
    return 1.0 - Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

def jaro_winkler_norm(s1, s2):
    return jellyfish.jaro_winkler_similarity(s1, s2)

def structural_vector_pair(left_str, right_str):
    """3-d vector: [Jaccard, Levenshtein, Jaro-Winkler] averaged across attributes
    shared by both entities within the pair. Domain-invariant (Eq. 7-8 of CIDER)."""
    left_attrs = parse_ditto_entity(left_str)
    right_attrs = parse_ditto_entity(right_str)
    common = set(left_attrs) & set(right_attrs)
    if not common:
        # Degenerate: no shared attributes → all zeros
        return np.zeros(3, dtype=np.float32)
    jaccs, levs, jaros = [], [], []
    for attr in common:
        l, r = str(left_attrs[attr]), str(right_attrs[attr])
        jaccs.append(jaccard_tokens(l, r))
        levs.append(levenshtein_norm(l, r))
        jaros.append(jaro_winkler_norm(l, r))
    return np.array([np.mean(jaccs), np.mean(levs), np.mean(jaros)], dtype=np.float32)

def structural_vectors_for_pairs(pairs):
    """Compute (N, 3) structural feature matrix."""
    return np.stack([structural_vector_pair(p['left'], p['right']) for p in pairs])

# Sanity check
sample = {'left': 'COL title VAL "canon powershot a3300" COL brand VAL "canon"',
          'right': 'COL title VAL "canon powershot elph 100" COL brand VAL "canon"'}
print(f"Sample structural vector: {structural_vector_pair(sample['left'], sample['right'])}")

# %% Active candidate source generation (Step 1)
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import KFold

def kfold_match_probabilities(X, y, k=KFOLDS):
    """For each source pair, return P(match | v) from K-fold Naive Bayes."""
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    probs = np.zeros(len(y), dtype=np.float64)
    for train_idx, val_idx in kf.split(X):
        nb = GaussianNB()
        nb.fit(X[train_idx], y[train_idx])
        probs[val_idx] = nb.predict_proba(X[val_idx])[:, 1]
    return probs

def binary_entropy(probs):
    """H = -p log2(p) - (1-p) log2(1-p), per Eq. 1."""
    p = np.clip(probs, 1e-10, 1 - 1e-10)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)

def cider_value(uncertainties, src_vectors, tgt_mean_vector, gamma=GAMMA):
    """Eq. 6: value(a,b) = entropy(a,b) + gamma * 1 / dist(v, v*)"""
    dists = np.linalg.norm(src_vectors - tgt_mean_vector, axis=1)
    inv_dists = 1.0 / (dists + 1e-10)
    return uncertainties + gamma * inv_dists

def select_candidate_source(source_pairs, source_vectors, target_vectors,
                             h=H_CANDIDATES, gamma=GAMMA):
    """Step 1 of CIDER. Returns indices of top-h candidate source pairs."""
    y = np.array([p['label'] for p in source_pairs], dtype=np.int64)
    # Uncertainty
    probs = kfold_match_probabilities(source_vectors, y)
    entropies = binary_entropy(probs)
    # Mean target vector
    v_star = target_vectors.mean(axis=0)
    # Value (Eq. 6)
    values = cider_value(entropies, source_vectors, v_star, gamma=gamma)
    # Top-h
    top_idx = np.argsort(-values)[:h]
    return top_idx, values

# %% In-context demo selection (Step 2) — STRATIFIED variant
# Deviation from CIDER paper: the paper uses pure top-K by similarity (Eq. 9).
# On extreme cross-domain pairs (product source -> citation target), pure top-K
# collapses to all-negative demos because (a) active candidate selection picks
# uncertainty=1.0 source pairs which are mostly hard negatives and (b) cross-
# domain semantic similarity is near 0, so structural similarity dominates.
# This stratified variant picks top-(K/2) positives + top-(K/2) negatives by
# similarity to guarantee class balance in the prompt.

def cosine_sim_rows(A, B):
    """Pairwise cosine similarity between rows of A (n, d) and rows of B (m, d).
    Returns (n, m)."""
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return A_norm @ B_norm.T

def cider_similarity_matrix(target_sem, candidate_sem, target_struct, candidate_struct,
                             alpha=ALPHA):
    """Eq. 9: sim = alpha * cos(sem) + (1-alpha) * cos(struct).
    Returns (n_target, n_candidate) matrix."""
    sem_sim = cosine_sim_rows(target_sem, candidate_sem)
    struct_sim = cosine_sim_rows(target_struct, candidate_struct)
    return alpha * sem_sim + (1 - alpha) * struct_sim

def select_demos_for_targets(sim_matrix, k=K_DEMOS, candidate_labels=None):
    """Stratified top-K demo selection. If candidate_labels is None, falls back
    to pure top-K (faithful to CIDER paper). If provided, picks top-(k/2)
    positives + top-(k/2) negatives by similarity per target row."""
    if candidate_labels is None:
        # Faithful CIDER: pure top-K by similarity
        return np.argsort(-sim_matrix, axis=1)[:, :k]

    # Stratified variant
    candidate_labels = np.asarray(candidate_labels)
    n_target, n_cand = sim_matrix.shape
    pos_idx = np.where(candidate_labels == 1)[0]
    neg_idx = np.where(candidate_labels == 0)[0]
    n_pos = k // 2
    n_neg = k - n_pos

    demos = np.zeros((n_target, k), dtype=np.int64)
    for i in range(n_target):
        # Top-n_pos positives by similarity
        if len(pos_idx) > 0:
            top_pos = pos_idx[np.argsort(-sim_matrix[i, pos_idx])[:n_pos]]
        else:
            top_pos = np.array([], dtype=np.int64)
        # Top-n_neg negatives by similarity
        if len(neg_idx) > 0:
            top_neg = neg_idx[np.argsort(-sim_matrix[i, neg_idx])[:n_neg]]
        else:
            top_neg = np.array([], dtype=np.int64)
        combined = np.concatenate([top_pos, top_neg])
        # Fallback: if one class was empty, fill remaining slots from any class
        if len(combined) < k:
            seen = set(combined.tolist())
            for cand_idx in np.argsort(-sim_matrix[i]):
                if cand_idx not in seen:
                    combined = np.append(combined, cand_idx)
                    seen.add(cand_idx)
                    if len(combined) == k:
                        break
        demos[i] = combined[:k]
    return demos

# %% Build CIDER prompt
def build_cider_prompt(target_pair, demo_pairs, source_domain, target_domain):
    """Per Fig. 3 of CIDER + explicit Yes/No instruction for LLaMA-3.3-70B
    (verbose-by-default; would otherwise produce explanatory text instead of
    a yes/no answer)."""
    lines = []
    src_cap = source_domain.capitalize()
    tgt_cap = target_domain.capitalize()
    for d in demo_pairs:
        lines.append(f"Do the two following {source_domain} descriptions refer to the same {source_domain}?")
        lines.append(f"{src_cap}1: {d['left']}")
        lines.append(f"{src_cap}2: {d['right']}")
        lines.append("Yes" if d['label'] == 1 else "No")
        lines.append("")
    lines.append(f"Do the two following {target_domain} descriptions refer to the same {target_domain}?")
    lines.append(f"{tgt_cap}1: {target_pair['left']}")
    lines.append(f"{tgt_cap}2: {target_pair['right']}")
    lines.append("Answer with Yes or No only.")   # required for LLaMA-3.3-70B
    return "\n".join(lines)

# Sanity check
demo_pair = {'left': 'COL title VAL canon a3300', 'right': 'COL title VAL canon elph 100', 'label': 0}
tgt_pair  = {'left': 'COL title VAL canon a3400', 'right': 'COL title VAL canon a3300', 'label': 1}
print(build_cider_prompt(tgt_pair, [demo_pair, demo_pair], "product", "product")[:700])

# %% Compute F1 + save metrics
def compute_metrics(predictions, labels):
    tp = sum(1 for p, l in zip(predictions, labels) if p == 1 and l == 1)
    fp = sum(1 for p, l in zip(predictions, labels) if p == 1 and l == 0)
    fn = sum(1 for p, l in zip(predictions, labels) if p == 0 and l == 1)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall

def save_metrics(out_dir, payload):
    import json
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(payload, f, indent=2)
    print(f"[wrote] {out_dir / 'metrics.json'}")


# ================================================================
# Instrumented llm_call — tracks prompt/completion tokens + API latency
# ================================================================
import time

TOKEN_STATS = {
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "api_latency_sec": 0.0,
    "n_calls": 0,
    "n_errors": 0,
}


def reset_token_stats():
    for k in TOKEN_STATS:
        TOKEN_STATS[k] = 0 if k != "api_latency_sec" else 0.0


def _snapshot_token_stats():
    return dict(TOKEN_STATS)


def llm_call(prompt, max_retries=3):
    """Single LLM call; returns text response. Accumulates tokens + latency in TOKEN_STATS."""
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            if PROVIDER in ("together", "openai"):
                resp = client.chat.completions.create(
                    model=MODEL,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=10,
                    temperature=0.0,
                )
                latency = time.perf_counter() - t0
                usage = getattr(resp, "usage", None)
                TOKEN_STATS["prompt_tokens"] += getattr(usage, "prompt_tokens", 0) if usage else 0
                TOKEN_STATS["completion_tokens"] += getattr(usage, "completion_tokens", 0) if usage else 0
                TOKEN_STATS["api_latency_sec"] += latency
                TOKEN_STATS["n_calls"] += 1
                return resp.choices[0].message.content.strip()
            elif PROVIDER == "anthropic":
                resp = client.messages.create(
                    model=MODEL,
                    max_tokens=10,
                    messages=[{"role": "user", "content": prompt}],
                )
                latency = time.perf_counter() - t0
                TOKEN_STATS["api_latency_sec"] += latency
                TOKEN_STATS["n_calls"] += 1
                return resp.content[0].text.strip()
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"  [llm error] {e}")
                TOKEN_STATS["n_errors"] += 1
                return ""
            time.sleep(2 ** attempt)
    return ""


def parse_yes_no(text):
    """Parse Yes/No (or Match/No match synonyms) from model response."""
    t = text.lower().strip()
    if t.startswith("yes") or t.startswith("match") or t.startswith("same"):
        return 1
    if t.startswith("no"):
        return 0
    if "yes" in t and "no" not in t:
        return 1
    if ("no match" in t) or ("not the same" in t) or ("different" in t and "same" not in t):
        return 0
    if "match" in t and "no match" not in t:
        return 1
    if "no" in t and "yes" not in t:
        return 0
    return 0

print("[ok] helpers loaded (data, SBERT, CIDER, prompt, LLM call with token tracking, metrics)")


In [ ]:
import sys, os
BASE = '/content/drive/MyDrive/cd-er-paradigm-choice'
print("cwd:", os.getcwd())
print("BASE env:", os.environ.get("REPO_ROOT"))
print("BASE in sys.path:", BASE in sys.path)
print("config/ exists:", os.path.isdir(f"{BASE}/config"))
print("config/__init__.py exists:", os.path.isfile(f"{BASE}/config/__init__.py"))
print("config/config.py exists:", os.path.isfile(f"{BASE}/config/config.py"))
try:
    import config.config
    print("import ok, BASE resolved to:", config.config.BASE)
except Exception as e:
    print("import failed:", type(e).__name__, e)


## 4. Load data with CIDER's 90/10 split

Assumes `get_source_train_and_target_test`, `encode_entity_pairs`, and helper functions are loaded (run the earlier setup cells first).

In [ ]:
import random
import numpy as np

source_train, target_test_full, source_dataset = get_source_train_and_target_test(REPRO_PAIR_TARGET)
print(f"source: {source_dataset} ({len(source_train)} pairs)")
print(f"target full: {len(target_test_full)} pairs")

random.seed(42)
indices = list(range(len(target_test_full)))
random.shuffle(indices)
n_val = int(len(target_test_full) * VAL_FRACTION)
val_idx = sorted(indices[:n_val])
test_idx = sorted(indices[n_val:])
target_val = [target_test_full[i] for i in val_idx]
target_test = [target_test_full[i] for i in test_idx]
print(f"val: {len(target_val)}, test: {len(target_test)}")

## 5. SBERT encode source + val + test (once)

In [ ]:
print("SBERT encoding source...")
src_sem = encode_entity_pairs(source_train)
print("SBERT encoding val...")
tgt_val_sem = encode_entity_pairs(target_val)
print("SBERT encoding test...")
tgt_test_sem = encode_entity_pairs(target_test)

## 6. Grid search on validation set + final test evaluation

In [ ]:
from pathlib import Path
import json, time

source_domain = DOMAIN_INFO.get(source_dataset, "entity")
target_domain = DOMAIN_INFO.get(REPRO_PAIR_TARGET, "entity")


def run_cider_at_hparams(gamma, alpha, target_sem, target_pairs, use_stratified=False):
    candidate_idx, _ = select_candidate_source(source_train, src_sem,
                                                 target_sem, h=REPRO_H, gamma=gamma)
    candidate_pairs = [source_train[i] for i in candidate_idx]
    candidate_sem = src_sem[candidate_idx]
    candidate_struct = structural_vectors_for_pairs(candidate_pairs)
    target_struct = structural_vectors_for_pairs(target_pairs)
    sim_matrix = cider_similarity_matrix(target_sem, candidate_sem,
                                          target_struct, candidate_struct, alpha=alpha)
    if use_stratified:
        candidate_labels = [p['label'] for p in candidate_pairs]
        demo_indices = select_demos_for_targets(sim_matrix, k=REPRO_K,
                                                  candidate_labels=candidate_labels)
    else:
        demo_indices = np.argsort(-sim_matrix, axis=1)[:, :REPRO_K]
    preds, labels = [], []
    for i, tp in enumerate(target_pairs):
        demos = [candidate_pairs[j] for j in demo_indices[i]]
        prompt = build_cider_prompt(tp, demos, source_domain, target_domain)
        resp = llm_call(prompt)
        preds.append(parse_yes_no(resp))
        labels.append(tp['label'])
    return compute_metrics(preds, labels)


print("Grid search on validation set...")
val_grid = {}
best_val_f1, best_gamma, best_alpha = 0.0, None, None
for gamma in GAMMA_GRID:
    for alpha in ALPHA_GRID:
        t = time.time()
        f1, _, _ = run_cider_at_hparams(gamma, alpha, tgt_val_sem, target_val,
                                           use_stratified=CIDER_STRATIFIED)
        val_grid[(gamma, alpha)] = f1
        print(f"  γ={gamma}, α={alpha}: val F1={f1:.4f} ({time.time()-t:.0f}s)")
        if f1 > best_val_f1:
            best_val_f1 = f1
            best_gamma, best_alpha = gamma, alpha

print(f"\nBest val: γ={best_gamma}, α={best_alpha}, F1={best_val_f1:.4f}")

# Reset token accumulator so we only count the final test-set run
reset_token_stats()
print(f"\nFinal test evaluation at γ={best_gamma}, α={best_alpha}...")
final_f1, final_p, final_r = run_cider_at_hparams(best_gamma, best_alpha,
                                                     tgt_test_sem, target_test,
                                                     use_stratified=CIDER_STRATIFIED)
print(f"Reproduction F1 = {final_f1:.4f}")

## 7. Verdict

In [ ]:
delta = final_f1 - CIDER_TARGET_F1
print("=" * 70)
print("CIDER FAITHFUL REPRODUCTION VERDICT")
print("=" * 70)
print(f"  CIDER paper (Table 5): F1 = {CIDER_TARGET_F1:.3f}")
print(f"  Our reproduction:      F1 = {final_f1:.4f}")
print(f"  Δ (ours - paper):      {delta:+.4f}")
if abs(delta) <= 0.03:
    verdict = "FAITHFUL"
    print("  → within ±3pp — IMPLEMENTATION FAITHFUL. Paper A negative results are credible.")
elif final_f1 < CIDER_TARGET_F1 - 0.03:
    verdict = "BELOW"
    print("  → materially BELOW paper. Investigate: prompt format, α/γ grid, SBERT choice.")
else:
    verdict = "ABOVE"
    print("  → materially ABOVE paper. Investigate: dev/test leakage, backbone version.")

# Save
out_path = Path(BASE) / "results" / "runs" / "cider_validation" / "co_wt_llama3-70b_deepinfra.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps({
    "backbone": REPRO_BACKBONE,
    "pair": f"{REPRO_PAIR_SOURCE} -> {REPRO_PAIR_TARGET}",
    "cider_paper_f1": CIDER_TARGET_F1,
    "our_reproduction_f1": final_f1,
    "test_precision": final_p,
    "test_recall": final_r,
    "delta_pp": delta,
    "prompt_tokens_total": TOKEN_STATS["prompt_tokens"],
    "completion_tokens_total": TOKEN_STATS["completion_tokens"],
    "api_latency_sec": TOKEN_STATS["api_latency_sec"],
    "n_api_calls": TOKEN_STATS["n_calls"],
    "n_api_errors": TOKEN_STATS["n_errors"],
    "verdict": verdict,
    "best_gamma": best_gamma,
    "best_alpha": best_alpha,
    "val_grid": {f"g{g}_a{a}": f for (g, a), f in val_grid.items()},
    "cider_stratified": CIDER_STRATIFIED,
    "hparams": {"h": REPRO_H, "k": REPRO_K, "kfolds": REPRO_KFOLDS,
                "val_fraction": VAL_FRACTION},
}, indent=2))
print(f"\n[saved] {out_path}")